# Backtester Prototype — run in Jupyter / Google Colab

This notebook clones `rohit-rakhecha/Backtester_Prototype` (branch `PR`), installs dependencies,
and runs the 8-stage AI Research Operating System pipeline end-to-end against the real NSE
factor-index data included in the repo. See `docs/PIPELINE.md` and `docs/DATA_SOURCES.md`
for the full write-up of what each stage does and why.

Works as-is in Google Colab. For a local Jupyter install, just run the cells in order —
the clone cell is a no-op if you already have the repo checked out locally and start
Jupyter from inside it.

## 1. Clone the repo (skips cleanly if it already exists)

In [ ]:
import os

REPO_URL = "https://github.com/rohit-rakhecha/Backtester_Prototype.git"
REPO_DIR = "Backtester_Prototype"
BRANCH = "PR"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists locally -- skipping clone. "
          f"cd into it and 'git pull' yourself if you want the latest commit.")

%cd {REPO_DIR}
!git status

## 2. Install dependencies

In [ ]:
%pip install -q -r requirements.txt
%pip install -q pypdf pytest matplotlib

## 3. Run the full 8-stage pipeline

This is the same script as `python examples/run_pipeline.py` from the command line --
running it with `%run` inside the notebook keeps every variable it defines (`card`,
`results`, `dataset`, `library`, ...) alive afterwards, so the cells below can explore
them further instead of just reading printed text.

In [ ]:
%run examples/run_pipeline.py

## 4. Plot cumulative performance (the part a plain script can't give you)

Uses the `results` dict and `bench_value` series that `run_pipeline.py` left in scope.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 6))
for name, res in results.items():
    ax.plot(res.value.index, res.value.values, label=name, linewidth=1.5)
ax.plot(bench_value.index, bench_value.values, label="NIFTY 500 (passive)", linewidth=1.5, linestyle="--", color="black")
ax.set_yscale("log")
ax.set_title("Cumulative portfolio value (log scale), net of India transaction costs")
ax.set_ylabel("Value (start = 1.0)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Inspect the audit trail and promotion ladder directly

In [ ]:
import pandas as pd

audit_entries = audit.read_all()
pd.DataFrame(audit_entries)[["gate", "card_id", "reviewer", "decision", "note", "timestamp"]]

In [ ]:
pd.DataFrame(library.history(card.card_id))

## 6. Run the test suite (optional sanity check)

In [ ]:
!python -m pytest tests/ -v

## 7. Try a different volatility target or strategy (interactive exploration)

Everything below reuses the SAME deterministic engine (`backtester.engine.portfolio`)
and cost model that Stage 05 used above -- only the signal function changes, which is
exactly the "AI only authors the signal" boundary the pipeline enforces.

In [ ]:
from backtester.engine import signals as sig
from backtester.engine.portfolio import PortfolioSimulator, performance_metrics
from backtester.engine.costs import DEFAULT_INDIA_COST_MODEL
import numpy as np

for tv in [0.08, 0.12, 0.16, 0.20]:
    fn = sig.markowitz(equal_weight, tv, l1_trust_region=1.0, cov_lookback_days=11,
                        one_way_cost_bps=one_way_bps, cash_annual_rate=0.0)
    sim = PortfolioSimulator(asset_returns=returns, cost_model=DEFAULT_INDIA_COST_MODEL,
                              cash_annual_rate=0.0, rebalance_frequency="ME")
    r = sim.run(fn)
    m = performance_metrics(r.value)
    print(f"target_vol={tv:>5.0%}  return={m['return']:.2%}  vol={m['volatility']:.2%}  "
          f"sharpe={m['sharpe']:.2f}  maxDD={m['max_drawdown']:.2%}")